<a href="https://colab.research.google.com/github/useDeep/learn_ml/blob/main/Play_around_with_hf_and_unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unsloth -v

/bin/bash: line 1: unsloth: command not found


In [ ]:
!pip install unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 48.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 22.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 MB 18.4 MB/s  0:00:14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 81.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.6 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled t

In [ ]:
!unsloth --version

unsloth 2026.11.3


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer= FastLanguageModel.from_pretrained(
    model_name= "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length= 2048,
    load_in_4bit= True
)

W1107 04:34:07.132000 206 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1107 04:34:07.202000 206 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth 2026.11.3: no GPU found, training on CPU (x86_64, 1 threads, bfloat16 = False). CPU training is much slower than GPU training; supported: text LoRA / full finetuning with SFTTrainer. Set UNSLOTH_DISABLE_CPU_FALLBACK=1 to raise instead.


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

... (3 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth (CPU): LlamaForCausalLM on 1 CPU threads. dtype = torch.float32, 4-bit = True, chunked CE = True.


In [ ]:
from transformers import TextStreamer

streamer= TextStreamer(tokenizer)

In [ ]:
text= "What are oranges to apples"
input_ids= tokenizer(text, return_tensors= "pt").to(model.device)
input_ids

{'input_ids': tensor([[128000,   3923,    527,  85138,    311,  41776]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [ ]:
output= model.generate(**input_ids, max_length= 20, streamer= streamer)
output

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|begin_of_text|>What are oranges to apples?
Oranges are not apples. They are two different types of fruits


tensor([[128000,   3923,    527,  85138,    311,  41776,   5380,   2244,   5637,
            527,    539,  41776,     13,   2435,    527,   1403,   2204,   4595,
            315,  26390]])

In [ ]:
tokenizer.decode(output)

['<|begin_of_text|>What are oranges to apples?\nOranges are not apples. They are two different types of fruits']

In [ ]:
from unsloth import get_chat_template

tokenizer= get_chat_template(tokenizer, chat_template= "llama-3.1")

In [ ]:
messages = [
    {"role": "user", "content": "What are oranges to apples?"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors = "pt",
    add_generation_prompt = True
).to(model.device)
input_ids

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,   5887,    220,   2366,     19,    271, 128009, 128006,
            882, 128007,    271,   3923,    527,  85138,    311,  41776,     30,
         128009, 128006,  78191, 128007,    271]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
output= model.generate(**input_ids, max_length= 60, streamer= streamer)
output

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What are oranges to apples?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Oranges and apples are two of the most widely consumed fruits in the world. Here's a


tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,   5887,    220,   2366,     19,    271, 128009, 128006,
            882, 128007,    271,   3923,    527,  85138,    311,  41776,     30,
         128009, 128006,  78191, 128007,    271,   2244,   5637,    323,  41776,
            527,   1403,    315,    279,   1455,  13882,  27073,  26390,    304,
            279,   1917,     13,   5810,    596,    264]])

In [ ]:
from datasets import load_dataset, Dataset

In [ ]:
data= [
    {
    "conversations": [
        {"from": "human", "value": "What is 2+2?"},
        {"from": "gpt", "value": "4"}
        ]
    }
  ]

In [ ]:
dataset= Dataset.from_list(data, split= "train")

In [ ]:
dataset

Dataset({
    features: ['conversations'],
    num_rows: 1
})

In [ ]:
model= FastLanguageModel.get_peft_model(
    model,
    r= 16,
    target_modules= ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
from unsloth import standardize_sharegpt

dataset= standardize_sharegpt(dataset)

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Unsloth: Standardizing formats (num_proc=1):   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
dataset[0]

{'conversations': [{'content': 'What is 2+2?', 'role': 'user'},
  {'content': '4', 'role': 'assistant'}]}

In [ ]:
dataset= dataset.map(
    lambda examples: {
        "text": [
            tokenizer.apply_chat_template(convo, tokenize= False)
            for convo in examples["conversations"]
        ]
    },
    batched= True
)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['conversations', 'text'],
    num_rows: 1
})

In [ ]:
dataset[0]

{'conversations': [{'content': 'What is 2+2?', 'role': 'user'},
  {'content': '4', 'role': 'assistant'}],
 'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is 2+2?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n4<|eot_id|>'}

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

args= TrainingArguments(
    per_device_train_batch_size= 2,
    gradient_accumulation_steps= 4,
    warmup_steps= 5,
    max_steps= 60,
    learning_rate= 2e-4,
    fp16= not torch.cuda.is_bf16_supported(),
    bf16= torch.cuda.is_bf16_supported(),
    logging_steps= 1,
    output_dir= "outputs"
)

trainer= SFTTrainer(
    model= model,
    train_dataset= dataset,
    dataset_text_field= "text",
    max_seq_length= 2048,
    args= args
)

Adding EOS to train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

In [23]:
trainer.train()

Step,Training Loss
1,5.134874
2,5.134874
3,4.647007
4,3.945471
5,3.329064
6,2.633235
7,1.862189
8,1.313053
9,1.011819
10,0.837773


Step,Training Loss
1,5.134874
2,5.134874
3,4.647007
4,3.945471
5,3.329064
6,2.633235
7,1.862189
8,1.313053
9,1.011819
10,0.837773


TrainOutput(global_step=60, training_loss=0.9489194581906001, metrics={'train_runtime': 1060.1011, 'train_samples_per_second': 0.453, 'train_steps_per_second': 0.057, 'total_flos': 15947576524800.0, 'train_loss': 0.9489194581906001, 'epoch': 60.0})

In [24]:
model.save_pretrained("finetuned_model")

In [25]:
tokenizer.pad_token

'<|finetune_right_pad_id|>'

In [26]:
tokenizer.pad_token_id

128004

In [27]:
tokenizer.pad_token_type_id

0

In [34]:
ftmodel, fttokenizer= FastLanguageModel.from_pretrained(
    model_name= "./finetuned_model",
    max_seq_length= 1024,
    load_in_4bit= True
)

Unsloth (CPU): ./finetuned_model is a LoRA adapter; loading its base unsloth/Llama-3.2-1B-Instruct first.


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth (CPU): LlamaForCausalLM on 1 CPU threads. dtype = torch.float32, 4-bit = True, chunked CE = True.


In [58]:
prompts= ["What is 8 + 3?"]

for prompt in prompts:
  infer_prompts= fttokenizer.apply_chat_template([{
      "role": "user",
      "content": prompt
  }], tokenize= False)

In [59]:
infer_prompts

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 07 Nov 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is 8 + 3?<|eot_id|>'

In [60]:
infer_inputs= fttokenizer(infer_prompts, return_tensors= "pt").to(ftmodel.device)

In [45]:
infer_outputs= ftmodel.generate(
    **infer_inputs,
    max_new_tokens= 514,
    temperature= 0.1,
    do_sample= True,
    pad_token_id= fttokenizer.pad_token_id
    )

infer_outputs

[transformers] Both `max_new_tokens` (=514) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[128000, 128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,
           2696,     25,   6790,    220,   2366,     18,    198,  15724,   2696,
             25,    220,   2589,   4723,    220,   2366,     21,    271, 128009,
         128006,    882, 128007,    271,   3923,    374,    220,     23,    489,
            220,     18,     30, 128009, 128006,  78191, 128007,    271,    806,
         128009]])

In [49]:
result= fttokenizer.batch_decode(infer_outputs, skip_special_tokens= True)

print(result[0])

system

Cutting Knowledge Date: December 2023
Today Date: 07 Nov 2026

user

What is 8 + 3?assistant

11


In [69]:
prompts= ["What is 8 + 3?", "what is the longest river in the southern part of the Democratic Republic of Korea also known as North Korea (in general)?"]
infer_prompts= []

for prompt in prompts:
  infer_prompts.append(fttokenizer.apply_chat_template([{
      "role": "user",
      "content": prompt
  }], tokenize= False))

In [70]:
infer_prompts

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 07 Nov 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is 8 + 3?<|eot_id|>',
 '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 07 Nov 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nwhat is the longest river in the southern part of the Democratic Republic of Korea also known as North Korea (in general)?<|eot_id|>']

In [72]:
infer_inputs= fttokenizer(
  infer_prompts,
  return_tensors= "pt",
  padding= True,
  truncation= True
).to(ftmodel.device)

In [73]:
infer_outputs= ftmodel.generate(
    **infer_inputs,
    max_new_tokens= 514,
    temperature= 0.1,
    do_sample= True,
    )

infer_outputs

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] Both `max_new_tokens` (=514) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[128000, 128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,
           2696,     25,   6790,    220,   2366,     18,    198,  15724,   2696,
             25,    220,   2589,   4723,    220,   2366,     21,    271, 128009,
         128006,    882, 128007,    271,   3923,    374,    220,     23,    489,
            220,     18,     30, 128009, 128004, 128004, 128004, 128004, 128004,
         128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004,
         128004, 128004,    271,   3923,    374,    220,     17,     10,     17,
         128009, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004,
         128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004, 128004,
         128004, 128004, 128004, 128004, 128004, 128004],
        [128000, 128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,
           2696,     25,   6790,    220,   2366,     18,    198,  15724,   2696,
             25,    220,   2589,   4723,    220,   

In [76]:
result1= fttokenizer.batch_decode(infer_outputs, skip_special_tokens= True)

print(result1[0])
print(result1[1])

system

Cutting Knowledge Date: December 2023
Today Date: 07 Nov 2026

user

What is 8 + 3?

What is 2+2
system

Cutting Knowledge Date: December 2023
Today Date: 07 Nov 2026

user

what is the longest river in the southern part of the Democratic Republic of Korea also known as North Korea (in general)?assistant

The longest river in the southern part of the Democratic People's Republic of Korea (North Korea) is the Choson River.


In [78]:
result2= fttokenizer.batch_decode(infer_outputs, skip_special_tokens= False)

print(result2[0])
print(result2[0])

<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Nov 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is 8 + 3?<|eot_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|>

What is 2+2<|eot_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune_right_pad_id|><|finetune